# 2 · Storing and analyzing attempts

I use DuckDB to keep the test inputs and predictions together. Each attempt has an ID that connects its metadata, four module records, and model output.

This notebook checks the schema, summarizes inputs with SQL, saves predictions, and exports results. It's also a way for me to get more comfortable with database work alongside the statistical model.

All current attempts are synthetic, and their actual scores are NULL. The evaluation query is in place, but there are no real score labels to evaluate yet.

One thing I want to improve is experiment history. Right now, rerunning the same attempt and model version updates the previous prediction. Separate run IDs would let me compare experiments without losing earlier outputs.

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project folder or in notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "satlab").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from satlab.database import connect, seed_demo, predict_attempt
from satlab.model import Settings

db = connect(ROOT / "data" / "learning.duckdb")
seed_demo(db)  # Safe to rerun: primary keys prevent duplicate fixture rows.
display(db.execute("SHOW TABLES").df())


## 1. Database structure

I separate source metadata in `attempts` from the four module rows in `module_inputs`.
A foreign key requires each module row to reference an existing attempt, while CHECK constraints enforce valid counts.
The `predictions` table stores model outputs together with their settings.

I save the settings with each prediction so I can check which assumptions produced it. The database is stored in a local DuckDB file.

In [ ]:
display(db.execute("DESCRIBE attempts").df())
display(db.execute("SELECT * FROM attempts ORDER BY attempt_id").df())
display(db.execute("SELECT * FROM module_inputs WHERE attempt_id = ? ORDER BY section, module", ["demo-1"]).df())


## 2. Summarizing the inputs

I group module records by attempt and section, then sum confident answers, multiple-choice guesses, and uncertain fill-ins. This provides a compact view of the inputs before score conversion.

`SELECT` specifies the result columns, `GROUP BY` defines each group, and `SUM` aggregates its rows.
I retain `NULL` for missing actual scores because an unknown score is different from zero.

In [ ]:
display(db.execute("""
    SELECT attempt_id, section,
           SUM(confident) AS confident_count,
           SUM(guess2 + guess3 + guess4) AS multiple_choice_guesses,
           SUM(fill_uncertain) AS uncertain_fill_ins
    FROM module_inputs
    GROUP BY attempt_id, section
    ORDER BY attempt_id, section
""").df())


## 3. Saving and joining predictions

For each attempt, I read the stored module inputs, call the Python model, and save the resulting interval in DuckDB. I then join those outputs to the attempt metadata using `attempt_id`.

The model version is `illustrative-v1`. Rerunning that version updates its previous output for the same attempt; the settings JSON records the latest assumptions. This keeps the latest result for that version. I plan to add separate run IDs to preserve earlier experiments.

I pass query values through `?` parameters instead of interpolating them into SQL strings.

In [ ]:
attempt_ids = [row[0] for row in db.execute("SELECT attempt_id FROM attempts ORDER BY attempt_id").fetchall()]
for attempt_id in attempt_ids:
    predict_attempt(db, attempt_id, Settings())

joined = db.execute("""
    SELECT a.attempt_id, a.student_id, a.test_form_id, a.is_synthetic,
           p.low, p.median, p.high, p.score_status
    FROM attempts AS a JOIN predictions AS p USING (attempt_id)
    ORDER BY a.attempt_id
""").df()
display(joined)


## 4. Exporting analysis results

I convert the joined query result to a DataFrame and export it to CSV. The CSV includes the synthetic-data flag and model status.

DuckDB remains the persistent data source. The same query result could later be serialized as JSON for an interface.

In [ ]:
output = ROOT / "artifacts"
output.mkdir(exist_ok=True)
joined.to_csv(output / "synthetic_predictions.csv", index=False)
print("Wrote", output / "synthetic_predictions.csv")


## 5. Evaluation query

I include a query for empirical coverage, mean interval width, and mean absolute error of the median. It excludes synthetic records and attempts with missing actual scores.

For the demonstration data, the result is **n = 0** with missing metrics. There are no actual score labels yet, so I can't use these records to measure accuracy.

Before applying this query to genuine evaluation data, I would separate training, calibration, and evaluation groups. Repeated students and shared test forms could otherwise leak information across splits. A nominal 90% interval covering only 60% of held-out scores would indicate undercoverage on that evaluation set.

In [ ]:
display(db.execute("""
    SELECT COUNT(*) AS n,
           AVG(CASE WHEN actual_rw + actual_math BETWEEN low AND high THEN 1.0 ELSE 0.0 END) AS coverage,
           AVG(high-low) AS mean_interval_width,
           AVG(ABS(median-(actual_rw+actual_math))) AS mean_absolute_error
    FROM attempts JOIN predictions USING (attempt_id)
    WHERE NOT is_synthetic AND actual_rw IS NOT NULL AND actual_math IS NOT NULL
""").df())
db.close()  # Release the file before opening it from another process.


Additional queries are collected in `sql/analysis.sql`. The final notebook presents a separate vector-search example that connects topic metadata, representations, and ranked results.

Reference: [DuckDB Python documentation](https://duckdb.org/docs/stable/clients/python/overview).